# Previsão de renda — análise reproduzível

Notebook de estudo alinhado com `modelagem.py` e com o aplicativo Streamlit. O notebook original `projeto-2.ipynb` foi preservado como histórico.


In [ ]:
import pandas as pd
from pathlib import Path
from modelagem import treinar_e_avaliar, importancias

base = pd.read_csv(Path("input") / "previsao_de_renda.csv")
print("Registros:", len(base))
print("Clientes únicos:", base["id_cliente"].nunique())
print("Datas de referência:", base["data_ref"].nunique())


## Validação por cliente

Separação por `GroupShuffleSplit`: ninguém é compartilhado entre treino e teste. Imputação e OneHotEncoder são aprendidos somente no treino, dentro do pipeline. A renda é transformada com log na modelagem e avaliada em reais.


In [ ]:
modelo, metricas, diagnosticos = treinar_e_avaliar(base)
pd.Series(metricas)


## Erros por faixa de renda real

As faixas usam a renda verdadeira (conhecida retrospectivamente). R² não equivale à acurácia.


In [ ]:
import numpy as np
faixas = diagnosticos.copy()
faixas["faixa"] = pd.cut(faixas["renda"], bins=[0, 2000, 5000, 10000, 20000, np.inf], labels=["Até 2 mil", "2–5 mil", "5–10 mil", "10–20 mil", "Acima de 20 mil"])
faixas.groupby("faixa", observed=True).agg(registros=("renda", "size"), mae=("erro_absoluto", "mean"), vies=("erro_assinado", "mean")).round(2)


## Importância das variáveis

São importâncias por redução de impureza, não efeitos causais. O modelo não deve orientar decisões financeiras ou de elegibilidade.


In [ ]:
importancias(modelo).head(15)
